In [1]:
import pandas as pd

df_jobs = pd.read_csv("../data/raw/marketing_sample_for_trulia_com-real_estate__20190901_20191031__30k_data.csv")

print(f"Total rows: {len(df_jobs)}")
print(f"Columns: {df_jobs.columns.tolist()}")
df_jobs.head(3)

Total rows: 30002
Columns: ['Job Title', 'Job Description', 'Job Type', 'Categories', 'Location', 'City', 'State', 'Country', 'Zip Code', 'Address', 'Salary From', 'Salary To', 'Salary Period', 'Apply Url', 'Apply Email', 'Employees', 'Industry', 'Company Name', 'Employer Email', 'Employer Website', 'Employer Phone', 'Employer Logo', 'Companydescription', 'Employer Location', 'Employer City', 'Employer State', 'Employer Country', 'Employer Zip Code', 'Uniq Id', 'Crawl Timestamp']


,Job Title,Job Description,Job Type,Categories,Location,City,State,Country,Zip Code,Address,...,Employer Phone,Employer Logo,Companydescription,Employer Location,Employer City,Employer State,Employer Country,Employer Zip Code,Uniq Id,Crawl Timestamp
0,Shift Manager,"<div id=""jobDescriptionText"" class=""jobsearch-...",NaN,NaN,"Mission Hills, CA 91345",Mission Hills,CA,United States,91345,NaN,...,NaN,https://d2q79iu7y748jz.cloudfront.net/s/_squar...,Del Taco is an American quick service restaura...,"Mission Hills, CA 91345",Mission Hills,CA,United States,91345,511f9a53920f4641d701d51d3589349f,2019-08-24 09:13:18 +0000
1,Operations Support Manager,"<div id=""jobDescriptionText"" class=""jobsearch-...",NaN,NaN,"Atlanta, GA 30342",Atlanta,GA,United States,30342,NaN,...,NaN,https://d2q79iu7y748jz.cloudfront.net/s/_logo/...,"Based in Atlanta, FOCUS Brands Inc. is an inno...",NaN,NaN,NaN,United States,NaN,4955daf0a3facbe2acb6c429ba394e6d,2019-09-19 08:16:55 +0000
2,Senior Product Manager - Data,"<div id=""jobDescriptionText"" class=""jobsearch-...",NaN,NaN,"Chicago, IL",Chicago,IL,United States,NaN,NaN,...,NaN,NaN,Vibes Corp. reputation was built and establish...,NaN,NaN,NaN,United States,NaN,a0e0d12df1571962b785f17f43ceae12,2019-09-18 02:13:10 +0000


In [2]:
# Keep only rows matching your CV pool's role types
keywords = "Java|Business Analyst|Project Manager|Scrum Master"

df_filtered = df_jobs[df_jobs["Job Title"].str.contains(keywords, case=False, na=False)].copy()

print(f"Matching job postings found: {len(df_filtered)}")
df_filtered["Job Title"].value_counts().head(20)

Matching job postings found: 545


Job Title
Project Manager                        71
Business Analyst                       21
Senior Business Analyst                10
Implementation Project Manager          9
Senior Project Manager                  9
IT Project Manager                      9
Technical Project Manager               8
Construction Project Manager            7
Business Operations Project Manager     5
IT Business Analyst                     4
Associate Project Manager               3
Project Manager II                      3
Field Project Manager                   3
Commercial Paint Project Manager        3
Senior Java Developer                   3
Java Software Engineer                  3
Salesforce Business Analyst             3
Sr. JAVA consultant                     3
Creative Project Manager                3
Sr Business Analyst                     3
Name: count, dtype: int64

In [3]:
# Exclude clearly non-IT/tech project manager roles
exclude_keywords = "Construction|Paint|Creative|Field Project"

df_relevant = df_filtered[~df_filtered["Job Title"].str.contains(exclude_keywords, case=False, na=False)].copy()

print(f"Relevant postings after exclusion: {len(df_relevant)}")
print(df_relevant["Job Title"].value_counts().head(25))

Relevant postings after exclusion: 525
Job Title
Project Manager                                           71
Business Analyst                                          21
Senior Business Analyst                                   10
Implementation Project Manager                             9
Senior Project Manager                                     9
IT Project Manager                                         9
Technical Project Manager                                  8
Business Operations Project Manager                        5
IT Business Analyst                                        4
Associate Project Manager                                  3
Project Manager II                                         3
Senior Java Developer                                      3
Java Software Engineer                                     3
Salesforce Business Analyst                                3
Sr. JAVA consultant                                        3
Sr Business Analyst                 

In [4]:
target_titles = [
    "Project Manager", "Senior Project Manager", "IT Project Manager",
    "Technical Project Manager", "Implementation Project Manager", "Associate Project Manager",
    "Business Analyst", "Senior Business Analyst", "IT Business Analyst",
    "Sr Business Analyst", "Business Analyst - Entry-Level", "Salesforce Business Analyst",
    "Senior Java Developer", "Java Software Engineer", "Sr. JAVA consultant",
    "Business Analyst/Project Manager", "Project Manager / Business Analyst", "Business Analyst Manager"
]

# Take the first occurrence of each title (to avoid duplicates skewing the sample)
selected_rows = []
for title in target_titles:
    match = df_relevant[df_relevant["Job Title"] == title].head(1)
    selected_rows.append(match)

df_offers = pd.concat(selected_rows).reset_index(drop=True)

print(f"Final selected job offers: {len(df_offers)}")
df_offers[["Job Title", "City", "State", "Company Name"]]

Final selected job offers: 18


,Job Title,City,State,Company Name
0,Project Manager,Boston,MA,American Well
1,Senior Project Manager,Philadelphia,PA,Razorfish Health
2,IT Project Manager,Fort Lee,NJ,Cross River Bank
3,Technical Project Manager,Auburn,NH,C Squared Systems LLC
4,Implementation Project Manager,Atlanta,GA,Serraview
5,Associate Project Manager,Shakopee,MN,Entrust Datacard
6,Business Analyst,Austin,TX,CesiumAstro
7,Senior Business Analyst,New Castle,PA,Ellwood Specialty Steel Company
8,IT Business Analyst,Costa Mesa,CA,ADC Solutions USA - Winplus
9,Sr Business Analyst,Tucson,AZ,"Sunquest Information Systems, Inc."


In [5]:
from bs4 import BeautifulSoup

def clean_html(raw_html):
    if pd.isna(raw_html):
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

df_offers["description_clean"] = df_offers["Job Description"].apply(clean_html)

# Preview the first one
print(df_offers["Job Title"].iloc[0])
print(df_offers["description_clean"].iloc[0][:500])

Project Manager
Company Description ------------------- At American Well, we believe digital care delivery will transform healthcare. Our mission is to connect and enable providers, insurers, patients, and innovators to deliver greater access to more affordable, higher quality care. We do this by partnering with our clients to deliver video visits over mobile and web. We have doctors, therapists, and specialists on staff to help people get care when and where they need it most. Brief Overview: The Project Manag


In [6]:
# Keep only the useful columns for our project
df_offers_final = df_offers[["Job Title", "Company Name", "City", "State", "Job Description", "description_clean"]].copy()
df_offers_final.columns = ["job_title", "company", "city", "state", "description_raw_html", "description_clean"]

df_offers_final.to_csv("../data/processed/job_offers.csv", index=False)
print(f"Saved {len(df_offers_final)} job offers to data/processed/job_offers.csv")
df_offers_final.head(3)

Saved 18 job offers to data/processed/job_offers.csv


,job_title,company,city,state,description_raw_html,description_clean
0,Project Manager,American Well,Boston,MA,"<div id=""jobDescriptionText"" class=""jobsearch-...",Company Description ------------------- At Ame...
1,Senior Project Manager,Razorfish Health,Philadelphia,PA,"<div id=""jobDescriptionText"" class=""jobsearch-...","Senior Project Manager- Razorfish Health, Phil..."
2,IT Project Manager,Cross River Bank,Fort Lee,NJ,"<div id=""jobDescriptionText"" class=""jobsearch-...","Who We Are At Cross River, we’re building the ..."


In [7]:
df_cvs = pd.read_csv("../data/processed/cvs_with_skills.csv")
df_offers = pd.read_csv("../data/processed/job_offers.csv")

print(f"CVs loaded: {len(df_cvs)}")
print(f"Job offers loaded: {len(df_offers)}")
print(df_cvs.columns.tolist())

CVs loaded: 224
Job offers loaded: 18
['filename', 'text', 'text_length', 'cleaned_text', 'extracted_skills', 'num_skills_found', 'years_experience', 'education']


In [8]:
import ast

df_cvs["extracted_skills"] = df_cvs["extracted_skills"].apply(ast.literal_eval)

# Quick check it worked
print(type(df_cvs["extracted_skills"].iloc[0]))
print(df_cvs["extracted_skills"].iloc[0][:5])

<class 'list'>
['java', 'core java', 'spring', 'spring mvc', 'hibernate']


In [10]:
# Reuse the same skills list and extraction function from notebook 01
# (we redefine them here since this is a separate notebook)

SKILLS_LIST = {
    "java": ["java", "j2ee", "core java", "spring", "spring boot", "spring mvc", "hibernate", "microservices", "multithreading"],
    "frontend": ["angular", "angularjs", "react", "reactjs", "javascript", "typescript", "html", "css", "bootstrap", "jquery", "node", "nodejs"],
    "database": ["sql", "mysql", "oracle", "postgresql", "mongodb", "nosql", "pl sql", "dynamodb"],
    "cloud_devops": ["aws", "azure", "gcp", "docker", "kubernetes", "jenkins", "ci cd", "git", "github", "terraform", "devops"],
    "api": ["rest", "restful", "soap", "api", "web service", "webservices", "graphql", "postman"],
    "testing": ["junit", "selenium", "mockito", "unit testing", "automation testing", "test driven development", "tdd"],
    "ba_tools": ["jira", "confluence", "visio", "sql", "excel", "power bi", "tableau", "sharepoint", "balsamiq"],
    "ba_skills": ["requirement gathering", "business requirement", "functional requirement", "gap analysis",
                  "use case", "user story", "stakeholder", "uat", "wireframe", "process improvement",
                  "business process", "data analysis", "root cause analysis", "swot"],
    "pm_methodology": ["agile", "scrum", "waterfall", "kanban", "sprint", "rup", "lean", "scaled agile", "safe agile"],
    "pm_certification": ["pmp", "csm", "cspo", "safe", "six sigma", "prince2", "itil", "capm"],
    "pm_tools": ["ms project", "jira", "asana", "trello", "confluence", "smartsheet"],
    "pm_skills": ["risk management", "budget management", "resource allocation", "project planning",
                  "project lifecycle", "vendor management", "cross functional", "roadmap"],
    "agile_roles": ["scrum master", "product owner", "agile coach", "release train engineer"],
    "agile_ceremonies": ["daily standup", "sprint planning", "sprint review", "retrospective",
                          "backlog grooming", "backlog refinement", "sprint retrospective"],
    "agile_concepts": ["user story", "story point", "velocity", "burndown chart", "epic",
                        "product backlog", "definition of done", "acceptance criteria",
                        "minimum viable product", "mvp", "scrum of scrums"],
    "education": ["bachelor", "master", "mba", "computer science", "information technology", "engineering degree"],
    "seniority": ["senior", "lead", "junior", "manager", "director", "years of experience", "years experience"],
    "work_auth": ["us citizen", "green card", "h1b", "opt", "cpt", "visa status", "work authorization"],
    "domain": ["healthcare", "finance", "banking", "insurance", "retail", "telecom", "supply chain", "ecommerce"],
    "soft_skills": ["leadership", "communication", "stakeholder management", "team management",
                     "problem solving", "negotiation", "collaboration", "time management",
                     "critical thinking", "adaptability", "mentoring", "presentation skill"]
}

all_skills = [skill for category in SKILLS_LIST.values() for skill in category]

def extract_skills(text, skills_list=all_skills):
    text = text.lower()
    found = []
    for skill in skills_list:
        skill_normalized = skill.replace(" of ", " ").replace(" the ", " ")
        if skill_normalized in text:
            found.append(skill)
    return found

# Apply to job offers' cleaned description
df_offers["extracted_skills"] = df_offers["description_clean"].apply(extract_skills)
df_offers["num_skills_found"] = df_offers["extracted_skills"].apply(len)

print("Skills extracted from job offers:")
df_offers[["job_title", "num_skills_found"]]

Skills extracted from job offers:


,job_title,num_skills_found
0,Project Manager,16
1,Senior Project Manager,12
2,IT Project Manager,10
3,Technical Project Manager,15
4,Implementation Project Manager,8
5,Associate Project Manager,7
6,Business Analyst,13
7,Senior Business Analyst,12
8,IT Business Analyst,11
9,Sr Business Analyst,11


In [ ]:
# Extracting years of experience required from job descriptions
import re

def extract_years_required(text):
    text = text.lower()
    pattern = r'(\d{1,2})\s*\+?\s*(?:year|yr)s?\b'
    matches = re.findall(pattern, text)
    if matches:
        numbers = [int(m) for m in matches if int(m) <= 45]
        if numbers:
            return min(numbers)  # for job offers, we take the MINIMUM (the stated requirement, not a stray higher number)
    return None

df_offers["years_required"] = df_offers["description_clean"].apply(extract_years_required)

print(df_offers[["job_title", "years_required"]])

                             job_title  years_required
0                      Project Manager             6.0
1               Senior Project Manager             7.0
2                   IT Project Manager             NaN
3            Technical Project Manager             NaN
4       Implementation Project Manager             5.0
5            Associate Project Manager             2.0
6                     Business Analyst             2.0
7              Senior Business Analyst             5.0
8                  IT Business Analyst             NaN
9                  Sr Business Analyst             5.0
10      Business Analyst - Entry-Level             2.0
11         Salesforce Business Analyst             5.0
12               Senior Java Developer             6.0
13              Java Software Engineer             NaN
14                 Sr. JAVA consultant             NaN
15    Business Analyst/Project Manager             7.0
16  Project Manager / Business Analyst             3.0
17        

In [12]:
#extracting required education from job offers
def extract_education(text):
    text = text.lower()
    if any(kw in text for kw in ["phd", "doctorate", "ph.d"]):
        return "PhD"
    elif any(kw in text for kw in ["mba", "master of business administration"]):
        return "MBA"
    elif any(kw in text for kw in ["master", "m.s.", "ms degree", "msc", "m.tech"]):
        return "Master"
    elif any(kw in text for kw in ["bachelor", "b.s.", "bs degree", "bsc", "b.tech", "b.e."]):
        return "Bachelor"
    elif any(kw in text for kw in ["associate degree", "a.s.", "diploma"]):
        return "Associate"
    else:
        return "Not specified"

df_offers["education_required"] = df_offers["description_clean"].apply(extract_education)

print(df_offers[["job_title", "education_required"]])

                             job_title education_required
0                      Project Manager      Not specified
1               Senior Project Manager           Bachelor
2                   IT Project Manager      Not specified
3            Technical Project Manager      Not specified
4       Implementation Project Manager           Bachelor
5            Associate Project Manager           Bachelor
6                     Business Analyst           Bachelor
7              Senior Business Analyst           Bachelor
8                  IT Business Analyst      Not specified
9                  Sr Business Analyst           Bachelor
10      Business Analyst - Entry-Level           Bachelor
11         Salesforce Business Analyst             Master
12               Senior Java Developer      Not specified
13              Java Software Engineer           Bachelor
14                 Sr. JAVA consultant           Bachelor
15    Business Analyst/Project Manager                MBA
16  Project Ma

In [13]:
df_offers.to_csv("../data/processed/job_offers.csv", index=False)
print("Saved updated job_offers.csv with skills, years_required, and education_required")

Saved updated job_offers.csv with skills, years_required, and education_required
